# 06 — App Architecture
### Harness, feedback loops, graceful degradation

---

**Format:** Narrated demo — pre-run outputs. The flow runs. Read what changed and why.

**The story:** Module 03 built a pipeline that works on good data with a responsive LLM. Production isn't that clean. This module adds three things that keep it running when it isn't:

```
Pattern              Metaflow tool        What it solves
───────────────────  ───────────────────  ──────────────────────────────────────────────
Graceful degradation @catch               One bad target doesn't kill the whole run
Eval-as-CI           evaluate step        Assertions run on every execution, fail loudly
Observability        @card                Per-target + run-level HTML reports, always
```

**The flow code from Module 03 is unchanged.** These are additive: new decorators on existing steps and one new step.

---
## Part 1 — Graceful degradation with `@catch`

Module 03 already has `@retry` on the `analyze` step. Retry handles transient failures — it tries again. But what happens when all retries are exhausted, or when the input data itself is broken?

Without `@catch`: the entire `foreach` run halts. Three targets succeed, one fails, you get nothing.

With `@catch`: the exception is caught, stored as an artifact named by the `var` parameter, and the flow continues. The next step can inspect it and act accordingly.

In [ ]:
# The key additions from Module 03 → Module 06
# (showing only what changed)

from metaflow import catch, retry, card, step, conda
from metaflow.cards import Markdown, Table, Artifact

# Before (Module 03):
#   @conda(...)
#   @retry(times=2)
#   @step
#   def ingest(self): ...

# After (Module 06) — @catch wraps @retry:
#   @conda(...)
#   @catch(var="ingest_error", print_exception=True)  ← new
#   @retry(times=2)
#   @step
#   def ingest(self): ...

# When ingest fails after all retries:
#   self.ingest_error = <the exception>   ← stored as artifact
#   flow continues to analyze             ← doesn't halt

# The analyze step checks for it:
def analyze_with_graceful_degradation(self):
    if hasattr(self, "ingest_error") and self.ingest_error is not None:
        # Record failure, produce a safe default result, continue
        self.result = {
            "classification":    "insufficient_data",
            "confidence":        0.0,
            "reasoning_summary": f"Skipped: ingest error — {self.ingest_error}",
        }
        return  # skip the LLM call, go to evaluate
    # ... normal analysis ...

print("@catch pattern loaded — no execution needed to understand it")
print("Key property: the run produces partial results rather than no results")

---
## Part 2 — Eval-as-CI: the `evaluate` step

The `evaluate` step runs after `analyze` on every execution. It calls assertion functions from `evals/assertions.py` — plain Python functions, no Metaflow dependency, testable with pytest.

Critical failures raise `AssertionError` — the step fails loudly, visibly, with a message pointing to the card. Non-critical failures become warnings stored as artifacts.

This is evals-as-CI: the same pattern as unit tests in a CI pipeline, applied to pipeline outputs.

In [ ]:
import sys
sys.path.insert(0, "..")
from evals.assertions import run_all_assertions

# Simulate what the evaluate step does with real pipeline outputs
# In the flow these come from self.report and self.result artifacts

# Mock outputs matching what Module 01 + Module 03 produce
from unittest.mock import MagicMock

mock_report = MagicMock()
mock_report.nulls          = {"PHASE": 0, "LC_DETREND": 0, "MODEL_INIT": 0}
mock_report.flux_std       = 0.00031482
mock_report.phase_range    = (-0.496, 0.496)
mock_report.duplicate_phases = 0
mock_report.model_dump     = lambda: {"flux_std": 0.00031482, "nulls": {}, "phase_range": (-0.496, 0.496)}

mock_result = {
    "classification":       "confirmed_transit",
    "confidence":           0.91,
    "transit_depth_pct":    1.0142,
    "reasoning_summary":    "The pipeline shows a clear periodic flux decrease consistent with a hot Jupiter transit. Transit depth and anomaly clustering are both consistent with WASP-18 b literature values.",
    "recommended_next_steps": ["Fit parametric model", "Compare against Sector 2"],
}

print("Running assertion suite against wasp18b results...\n")

eval_result = run_all_assertions(mock_report, mock_result)

for check in eval_result["results"]:
    icon = "✓" if check["passed"] else ("✗" if check["critical"] else "⚠")
    print(f"  {icon}  {check['name']:<30}{check['detail']}")

print(f"\n{eval_result['n_passed']} passed · "
      f"{eval_result['n_failed']} failed · "
      f"{eval_result['n_warnings']} warnings")
print(f"eval_result.passed = {eval_result['passed']}")

Running assertion suite against wasp18b results...

  ✓  no_nulls                  Total nulls: 0
  ✓  flux_std_reasonable       flux_std=0.00031482 (expected 1e-06–0.01)
  ✓  phase_range_sensible      phase_range=(-0.496, 0.496), span=0.992
  ✓  no_duplicate_phases       duplicate_phases=0
  ✓  classification_valid      classification='confirmed_transit'
  ✓  confidence_in_range       confidence=0.91
  ✓  reasoning_present         reasoning_summary length=142
  ✓  transit_depth_physical    transit_depth_pct=1.0142% (expected 0.01–5.0%)

8 passed · 0 failed · 0 warnings
eval_result.passed = True


In [ ]:
# Now show what happens with a bad target
print("Simulating a bad target: malformed data + confused agent...\n")

bad_report = MagicMock()
bad_report.nulls             = {"PHASE": 0, "LC_DETREND": 0, "MODEL_INIT": 0}
bad_report.flux_std          = 0.00000042   # suspiciously flat
bad_report.phase_range       = (-0.496, 0.496)
bad_report.duplicate_phases  = 0
bad_report.model_dump        = lambda: {}

bad_result = {
    "classification":    "confirmed_transit",
    "confidence":        1.7,       # out of range — model returned garbage
    "transit_depth_pct": 0.0001,    # implausibly shallow
    "reasoning_summary": "OK",      # too short
}

bad_eval = run_all_assertions(bad_report, bad_result)

for check in bad_eval["results"]:
    icon = "✓" if check["passed"] else ("✗" if check["critical"] else "⚠")
    print(f"  {icon}  {check['name']:<30}{check['detail']}")

print(f"\n{bad_eval['n_passed']} passed · "
      f"{bad_eval['n_failed']} failed · "
      f"{bad_eval['n_warnings']} warnings")
print(f"eval_result.passed = {bad_eval['passed']}")
print(f"critical_failures: {bad_eval['critical_failures']}")
print("\nIn the flow this raises AssertionError — visible failure, not silent wrong answer.")

Simulating a bad target: malformed data + confused agent...

  ✓  no_nulls                  Total nulls: 0
  ✗  flux_std_reasonable       flux_std=0.00000042 (expected 1e-06–0.01)
  ✓  phase_range_sensible      phase_range=(-0.496, 0.496), span=0.992
  ✓  no_duplicate_phases       duplicate_phases=0
  ✓  classification_valid      classification='confirmed_transit'
  ✗  confidence_in_range       confidence=1.7
  ⚠  reasoning_present         reasoning_summary length=3
  ⚠  transit_depth_physical    transit_depth_pct=0.0001% (expected 0.01–5.0%)

4 passed · 2 failed · 2 warnings
eval_result.passed = False
critical_failures: ['flux_std_reasonable', 'confidence_in_range']

In the flow this raises AssertionError — visible failure, not silent wrong answer.


---
## Part 3 — Observability with `@card`

Cards are HTML reports attached to steps. They're generated after the step completes and stored alongside artifacts — you can view them any time, even for runs from weeks ago.

Two cards in the harnessed flow:
- `@card` on `evaluate` → per-target report (assertions table, classification, validation stats)
- `@card` on `end` → run-level summary (all targets, pass/fail, errors)

Cards don't change flow behavior. If card generation fails, the flow continues.

In [ ]:
# How @card is used in the evaluate step (simplified)
from metaflow.cards import Markdown, Table, Artifact

# In the flow, current.card is available inside @card-decorated steps
# current.card.append() adds a row to the card

def build_evaluate_card_example(target, eval_result, result, report):
    """
    This is what runs inside the @card(type='blank') evaluate step.
    Shown here as a plain function so you can read it without running the flow.
    """
    from metaflow import current

    status_icon = "✓" if eval_result["passed"] else "✗"
    current.card.append(Markdown(f"## {status_icon} {target} — Eval Report"))

    # Summary line
    ev = eval_result
    current.card.append(Markdown(
        f"**{ev['n_passed']} passed** · {ev['n_failed']} failed · {ev['n_warnings']} warnings"
    ))

    # Agent result
    current.card.append(Markdown(
        f"**Classification:** {result['classification']}  \n"
        f"**Confidence:** {result['confidence']:.2f}  \n"
        f"**Transit depth:** {result['transit_depth_pct']:.4f}%"
    ))

    # Assertions as a table
    rows = [
        [Markdown(f"`{c['name']}`"),
         Markdown("✓" if c["passed"] else ("✗" if c["critical"] else "⚠")),
         Markdown(c["detail"])]
        for c in ev["results"]
    ]
    current.card.append(Table(rows, headers=["Assertion", "Status", "Detail"]))

    # Raw ValidationReport for debugging
    current.card.append(Artifact(report.model_dump()))

print("Card structure defined — runs inside @card step during flow execution")

---
## Part 4 — Run the harnessed flow

Three targets: two good, one deliberately broken.

In [ ]:
# python flows/harnessed_lightcurve_flow.py run --targets wasp18b,wasp12b,bad_target

# Pre-run output — the key lines to notice:
#   bad_target exhausts retries → @catch stores the error → flow continues
#   wasp18b and wasp12b complete successfully
#   evaluate/bad_target records the failure but doesn't raise (ingest_failed is not a critical assertion)
#   Run summary: 2/3 passed — partial results, not zero results

print("""Metaflow 2.18.4 executing HarnessedLightcurveFlow...
[start] Processing 3 targets: ['wasp18b', 'wasp12b', 'bad_target']

[ingest/wasp18b]   ValidationReport: nulls=0, flux_std=0.00031482
[ingest/wasp12b]   ValidationReport: nulls=0, flux_std=0.00028891
[ingest/bad_target] ERROR: FileNotFoundError: bad_target_lightcurve.csv not found
[ingest/bad_target] Retrying (1/2)...
[ingest/bad_target] ERROR: FileNotFoundError: bad_target_lightcurve.csv not found
[ingest/bad_target] Retrying (2/2)...
[ingest/bad_target] ERROR: FileNotFoundError: bad_target_lightcurve.csv not found
[ingest/bad_target] @catch: stored exception in self.ingest_error — continuing

[analyze/wasp18b]   classification=confirmed_transit, confidence=0.91
[analyze/wasp12b]   classification=confirmed_transit, confidence=0.88
[analyze/bad_target] Skipping — ingest failed: bad_target_lightcurve.csv not found

[evaluate/wasp18b]   8/8 assertions passed ✓
[evaluate/wasp12b]   8/8 assertions passed ✓
[evaluate/bad_target] 0 critical failures, 1 failure recorded (ingest_failed)

[join] Merged 3 targets

Run summary: 2/3 passed
  ✓ wasp18b: 0 critical failures
  ✓ wasp12b: 0 critical failures
  ✗ bad_target: 1 critical failures

Done! ✓  (flow completed despite bad_target failure)""")

Metaflow 2.18.4 executing HarnessedLightcurveFlow...
[start] Processing 3 targets: ['wasp18b', 'wasp12b', 'bad_target']

[ingest/wasp18b]   ValidationReport: nulls=0, flux_std=0.00031482
[ingest/wasp12b]   ValidationReport: nulls=0, flux_std=0.00028891
[ingest/bad_target] ERROR: FileNotFoundError: bad_target_lightcurve.csv not found
[ingest/bad_target] Retrying (1/2)...
[ingest/bad_target] ERROR: FileNotFoundError: bad_target_lightcurve.csv not found
[ingest/bad_target] Retrying (2/2)...
[ingest/bad_target] ERROR: FileNotFoundError: bad_target_lightcurve.csv not found
[ingest/bad_target] @catch: stored exception in self.ingest_error — continuing

[analyze/wasp18b]   classification=confirmed_transit, confidence=0.91
[analyze/wasp12b]   classification=confirmed_transit, confidence=0.88
[analyze/bad_target] Skipping — ingest failed: bad_target_lightcurve.csv not found

[evaluate/wasp18b]   8/8 assertions passed ✓
[evaluate/wasp12b]   8/8 assertions passed ✓
[evaluate/bad_target] 0 critica

---
## Part 5 — Inspecting results and cards

Cards and the Client API both work the same whether the flow ran locally, on a Linux box, or on Outerbounds.

In [ ]:
# Inspect the last run with the Client API
# (works identically locally, on Linux box, or via Outerbounds)

# from metaflow import Flow
# run = Flow('HarnessedLightcurveFlow').latest_run
# print(run.data.eval_summary)
# print(run.data.all_results['wasp18b']['result'])

# Pre-run output:
print("""Latest run: HarnessedLightcurveFlow/1745590412

eval_summary:
  wasp18b:    passed=True,  n_failed=0
  wasp12b:    passed=True,  n_failed=0
  bad_target: passed=False, n_failed=1

wasp18b result:
  classification:    confirmed_transit
  confidence:        0.91
  transit_depth_pct: 1.0142%""")

Latest run: HarnessedLightcurveFlow/1745590412

eval_summary:
  wasp18b:    passed=True,  n_failed=0
  wasp12b:    passed=True,  n_failed=0
  bad_target: passed=False, n_failed=1

wasp18b result:
  classification:    confirmed_transit
  confidence:        0.91
  transit_depth_pct: 1.0142%


In [ ]:
# How to view cards — three ways
print("""# View cards from the command line:

# Run-level summary (end step):
python flows/harnessed_lightcurve_flow.py card view end

# Per-target eval card (evaluate step, wasp18b task):
python flows/harnessed_lightcurve_flow.py card view evaluate

# Cards are stored as artifacts — available for any past run:
python flows/harnessed_lightcurve_flow.py card view end --run-id 1745590412

# In a notebook:
from metaflow import Flow
run = Flow('HarnessedLightcurveFlow').latest_run
end_step = run['end']
cards = end_step.task.cards
cards[0].get()   # returns the HTML""")

# View cards from the command line:

# Run-level summary (end step):
python flows/harnessed_lightcurve_flow.py card view end

# Per-target eval card (evaluate step, wasp18b task):
python flows/harnessed_lightcurve_flow.py card view evaluate

# Cards are stored as artifacts — available for any past run:
python flows/harnessed_lightcurve_flow.py card view end --run-id 1745590412

# In a notebook:
from metaflow import Flow
run = Flow('HarnessedLightcurveFlow').latest_run
end_step = run['end']
cards = end_step.task.cards
cards[0].get()   # returns the HTML


---
## What changed from Module 03 — summary

| | Module 03 | Module 06 |
|---|---|---|
| Bad target behavior | Entire run halts | Bad target recorded, others complete |
| LLM failure after retries | Entire run halts | Failure stored in artifact, flow continues |
| Output validation | None | 8 assertions, critical failures raise |
| Silent wrong answers | Possible | Caught by confidence/classification assertions |
| Run visibility | Stdout logs only | Cards on evaluate + end steps |
| Debugging past runs | Re-run to reproduce | Client API + cards available for every run |
| Agent code | unchanged | unchanged |
| `ingestion.py` | unchanged | unchanged |
| Pydantic models | unchanged | unchanged |

The three additions (`@catch`, `evaluate` step, `@card`) make the difference between a pipeline that works on your laptop and one that runs in production without someone watching it.

**Next:** [Module 07 — Mission-Critical Infrastructure](../07-mission-critical-infrastructure/) — supply chain security, CVE scanning, `conda-lock`, `conda-pack`, and air-gapped deployment. The pipeline continues to not change.